# PA operating sweep — ADRV9026

TX1 into ORx1 on LO1. For each signal the waveform is loaded once, then LO1 is
retuned across `FREQS_HZ`. Each pair runs its own compression search; the
attenuation that hits the target is that pair's lock.

From the lock, one capture per backoff (more attenuation, less TX power). The
ORx AGC runs again before each one. Each capture is two periods; the aligned
period is saved. The reference is written once per signal (`<signal>_in.txt`)
and every capture adds one row to `TX1_conditions.csv`.

This CSV is what `dpd_replay.ipynb` and `dpd_linearize_loop.ipynb` read.

## 1. Parameters

In [ ]:
from adrvtrx import TxChannel, RxChannel

PROFILE = "ADRV9025Init_StdUseCase98_LinkSharing.profile"

TX_CHANNEL  = TxChannel.TX1
ORX_CHANNEL = RxChannel.ORX1

FREQS_HZ = [2_000_000_000, 2_200_000_000, 2_400_000_000, 2_600_000_000, 2_800_000_000]
SIGNALS = [
    (40,  "new40MHz256QAM_CFRed.txt"),
    (100, "new100MHz256QAM_CFRed.txt"),
]
BACKOFFS_DB = [0, 1, 2, 3, 4, 5]

TX_ATTEN_START_DB = 15.0
TX_ATTEN_MIN_DB   = 7.0
COARSE_STEP_DB    = 5.0
FINE_STEP_DB      = 0.2

TARGET_COMPRESSION_DB = 3.0
COMP_TOL_DB           = 0.1
WINDOW_MS             = 0.1

ORX_TARGET_DBFS = -1.0
ORX_TOL_UP_DB   = 0.3
ORX_TOL_DOWN_DB = 0.6

OUTPUT_OVERSAMPLE = 2
SAVE_DIR = "captures/operating_sweep"
CSV_NAME = "TX1_conditions.csv"

## 2. Imports, config, profile

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from adrvtrx.config import load_config
from adrvtrx.profile import read_profile
from adrvtrx.radio import Radio
from adrvtrx.experiment import verify_status
from adrvtrx.waveform import full_scale

from adrvtrx.waveform import load_tab_iq, prepare_tx
from adrvtrx.transmit import transmit_bands
from adrvtrx.capture import autolevel_capture
from adrvtrx.compression import find_compression_point
from adrvtrx.conditions import (
    ConditionLog, OperatingCondition, capture_point, condition_name, save_aligned_iq,
)
from adrvtrx.metrics import papr_db
from adrvtrx.radio import MAX_TX_ATTEN_DB

cfg = load_config()
cfg.profile_name = PROFILE
cfg.lo.lo1_hz = int(FREQS_HZ[0])

info = read_profile(cfg.profile_path)
FS = info.orx_rate_hz
print("Profile:", cfg.profile_path.name)
print(f"TX  {info.tx_rate_khz/1000:.2f} MSPS, {info.tx_bits} bit (FS {full_scale(info.tx_bits)})")
print(f"ORx {info.orx_rate_khz/1000:.2f} MSPS, {info.rx_bits} bit (FS {full_scale(info.rx_bits)})")
print(f"{len(SIGNALS)} signals x {len(FREQS_HZ)} frequencies x {len(BACKOFFS_DB)} backoffs"
      f" = {len(SIGNALS) * len(FREQS_HZ) * len(BACKOFFS_DB)} captures")

## 3. Connect and program

One program for the whole grid. Later frequencies are `retune_lo` only.

In [ ]:
radio = Radio(cfg)
radio.connect()
radio.force_safe()
radio.program()

radio.set_tx_atten(TX_CHANNEL, TX_ATTEN_START_DB)
lock = radio.retune_lo("LO1", int(FREQS_HZ[0]))

for k, v in verify_status(radio).items():
    print(f"{k}: {v}")
print(f"LO1 = {radio.get_lo('LO1')/1e6:.3f} MHz   lock 0x{lock:X}")
print(f"TX1 atten = {radio.get_tx_atten(TX_CHANNEL):.2f} dB")

## 4. Sweep

At each frequency: transmit, search from the start attenuation, then one
capture per backoff with the AGC before each. TX stops before the next
frequency and whenever the cell ends. Each CSV row is flushed as it is written.

In [ ]:
def show(rec):
    print(f"  atten {rec['atten_db']:5.2f} dB  gain {rec['orx_gain']:3d}  "
          f"peak {rec['peak_dbfs']:+5.1f} dBFS  railed {rec['railed']:<4d}  "
          f"comp {rec['compression_db']:5.2f} dB  {rec['action']}")

save_dir = Path(SAVE_DIR)
save_dir.mkdir(parents=True, exist_ok=True)
n_total = len(SIGNALS) * len(FREQS_HZ) * len(BACKOFFS_DB)
n_saved = 0

with ConditionLog(save_dir / CSV_NAME) as log:
    try:
        for bw_mhz, signal_path in SIGNALS:
            iq = load_tab_iq(signal_path)
            i_tx, q_tx = prepare_tx(iq, info.tx_bits)
            ref = i_tx.astype(np.float64) + 1j * q_tx.astype(np.float64)
            capture_ms = OUTPUT_OVERSAMPLE * len(ref) / FS * 1e3
            in_path = save_dir / f"{Path(signal_path).stem}_in.txt"
            save_aligned_iq(ref, in_path, info.tx_bits)
            print(f"\n{Path(signal_path).name}  N={len(iq)}  PAPR {papr_db(iq):.2f} dB")

            for lo_hz in FREQS_HZ:
                print(f"\n  {bw_mhz} MHz @ {lo_hz / 1e6:.0f} MHz")
                radio.set_tx_atten(TX_CHANNEL, TX_ATTEN_START_DB)
                radio.retune_lo("LO1", int(lo_hz))
                transmit_bands(radio, {TX_CHANNEL: iq}, info.tx_bits)

                result = find_compression_point(
                    radio, TX_CHANNEL, ORX_CHANNEL, ref,
                    rx_bits=info.rx_bits, fs=FS,
                    target_compression_db=TARGET_COMPRESSION_DB,
                    start_atten_db=TX_ATTEN_START_DB, atten_min_db=TX_ATTEN_MIN_DB,
                    window_s=WINDOW_MS * 1e-3, oversample=OUTPUT_OVERSAMPLE,
                    coarse_step_db=COARSE_STEP_DB, fine_step_db=FINE_STEP_DB,
                    comp_tol_db=COMP_TOL_DB, target_dbfs=ORX_TARGET_DBFS,
                    tol_up_db=ORX_TOL_UP_DB, tol_down_db=ORX_TOL_DOWN_DB,
                    on_step=show,
                )
                locked = result.final_atten_db
                print(f"  lock {locked:.2f} dB  compression {result.compression_db:.2f} dB  "
                      f"converged={result.converged}  {result.reason}")

                for backoff in BACKOFFS_DB:
                    atten = min(locked + backoff, MAX_TX_ATTEN_DB)
                    radio.set_tx_atten(TX_CHANNEL, atten)
                    agc = autolevel_capture(
                        radio, ORX_CHANNEL, bits=info.rx_bits, orx_rate_hz=FS,
                        target_dbfs=ORX_TARGET_DBFS, tol_up_db=ORX_TOL_UP_DB,
                        tol_down_db=ORX_TOL_DOWN_DB, verify_capture_ms=capture_ms,
                    )
                    point = capture_point(
                        radio, ORX_CHANNEL, ref, rx_bits=info.rx_bits, fs=FS,
                        bw_hz=bw_mhz * 1_000_000, oversample=OUTPUT_OVERSAMPLE,
                    )
                    name = condition_name(TX_CHANNEL, backoff, lo_hz, bw_mhz)
                    out_path = save_dir / f"{name}_out.txt"
                    save_aligned_iq(point.y_aligned, out_path, info.rx_bits)
                    log.append(OperatingCondition(
                        name=name, signal=Path(signal_path).name, freq_hz=int(lo_hz),
                        bw_mhz=bw_mhz, backoff_db=backoff, locked_atten_db=locked,
                        tx_atten_db=atten, orx_gain=agc.final_gain_index,
                        compression_db=result.compression_db, converged=result.converged,
                        peak_dbfs=point.clip.peak_dbfs, railed=point.clip.railed_samples,
                        in_file=in_path.name, out_file=out_path.name, **point.metrics,
                    ))
                    n_saved += 1
                    m = point.metrics
                    print(f"  [{n_saved}/{n_total}] {name}  atten {atten:.2f} dB  "
                          f"gain {agc.final_gain_index}  NMSE {m['nmse_db']:.2f} dB  "
                          f"ACLR {m['aclr_lower_dbc']:.2f}/{m['aclr_upper_dbc']:.2f} dBc")

                radio.disable_tx()
                print("  TX stopped")
    finally:
        radio.disable_tx()

print(f"\n{n_saved} captures in {save_dir}   CSV {log.path}")

## Safe-state and disconnect

In [ ]:
radio.safe_state()
radio.disconnect()
print("TX safe, board disconnected")

## 6. NMSE and ACLR versus backoff (from the CSV)

In [ ]:
from adrvtrx.conditions import load_conditions

rows = load_conditions(Path(SAVE_DIR) / CSV_NAME)
keys = sorted({(r.bw_mhz, r.freq_hz) for r in rows})
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for bw, freq in keys:
    grp = sorted((r for r in rows if r.bw_mhz == bw and r.freq_hz == freq),
                 key=lambda r: r.backoff_db)
    b = [r.backoff_db for r in grp]
    label = f"{bw} MHz @ {freq / 1e6:.0f} MHz"
    ax[0].plot(b, [r.nmse_db for r in grp], "o-", label=label)
    ax[1].plot(b, [max(r.aclr_lower_dbc, r.aclr_upper_dbc) for r in grp], "o-", label=label)
ax[0].set_ylabel("NMSE (dB)"); ax[1].set_ylabel("worst ACLR (dBc)")
for a in ax:
    a.set_xlabel("backoff (dB)"); a.grid(True)
ax[1].legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()